# Workspace for Fourier Tasks

- By: (redacted), 2025
- Corresponding paper (redacted), 2026
- Licence: MIT
- [Click here for the Fourier source repository (redacted)](https://fake.url)
- [Click here for the implementations on AMADS](https://github.com/music-computing/amads/)

## Background

The DFT magnitudes (spectrum) of a set
are equivalent to that of the set's:
- rotation (any step size) (= pitch 'transposition'),
- mirror image (= pitch 'inversion'),
- complement,
- scalar multiple.


## Task

- Type: Implement
- Task:
  - Take any vector, such as the binary PCP vector for (the scale of) C major `[1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1]`.
  - Write a function to calculate the DFT of this set and demonstrate the various equivalences mentioned above:
    1. First, make a new set by rotating the original (equivalent to musical transposition).
       - Hint: index `0:m` and `m:n` and reverse, where $1 < m < n$.
    2. Next reverse the set. Hint: `[::-1]`.
    3. Then, take the complement. Hint: map 0 to 1 and 1 to 0.
    4. Finally, proportionally increase the amplitudes, e.g., by replacing the 1s with 2s (double all the values).
  - In each case, verify that the altered form returns the same absolute DFT values (again, magnitudes, not phase).
  - Bonus: try your function on several different source sets and verify that they are _not_ equivalent to each other.
- Reference implementation:
    - `from amads.core.vectors_sets import scalar_multiply`
    - `from amads.core.vector_transforms_checks import rotate, mirror, complement`


## Workspace

## Reference 1: Repetition

In [ ]:
import numpy as np
def fourier_magnitudes(input) -> np.array:
    """Convenience function to ensure consistent handling of Fourier magnitude."""
    return np.absolute(np.fft.fft(input))

In [ ]:
one_tresillo = (1, 0, 0, 1, 0, 0, 1, 0)
one_tresillo_magnitudes = fourier_magnitudes(one_tresillo)
len(one_tresillo) == len(one_tresillo_magnitudes)

In [ ]:
num_repeats = 2
repeat_tresillo = one_tresillo * num_repeats
repeat_tresillo

In [ ]:
repeat_tresillo_magnitudes = fourier_magnitudes(repeat_tresillo)
len(repeat_tresillo_magnitudes) == len(one_tresillo) * num_repeats

In [ ]:
np.allclose(one_tresillo_magnitudes, repeat_tresillo_magnitudes[::num_repeats] / num_repeats)

In [ ]:
# Generalise

def demonstrate_repeat_mapping(
        vector: tuple,
        num_repeats: int = 4
) -> bool:
    """
    Demonstrates the slightly different case of repetition.
    """
    before = fourier_magnitudes(vector)
    after = fourier_magnitudes(vector * num_repeats)
    assert np.allclose(before, after[::num_repeats] / num_repeats)
    return True

In [ ]:
demonstrate_repeat_mapping(one_tresillo, num_repeats=4)

## Reference 2: "rotate", "mirror", "multiply", "complement"

In [ ]:
operations = ["rotate", "mirror", "multiply", "complement"]  # NB: not repeat.

from amads.core.vectors_sets import scalar_multiply
from amads.core.vector_transforms_checks import rotate, mirror, complement

def equivalence_comparison(
        vector: tuple[int, ...],
        operation: str = "rotation"
) -> bool:
    """
    Creates a modified form of a set and checks equivalence.

    Check this (from scratch) against built in `np.eye`.

    :param vector: the input data to test.
    :param operation: the change to make. Chose from
        "rotate" (e.g., pitch transpose),
        "mirror" (e.g., rhythm retrograde transpose for pitch),
        "multiply" (amplitudes),
        "complement".
        Note that repeat is handled separately.

    return: bool
    """

    before = fourier_magnitudes(vector)

    if operation == "rotate":
        after = rotate(vector)
    elif operation == "mirror":
        after = mirror(vector, index_of_symmetry=0)
    elif operation == "multiply":
        after = scalar_multiply(vector, scale_factor=2)
    elif operation == "complement":
        after = complement(vector)
    else:
        raise ValueError(f"Invalid operation: must be one of {operations}")

    after = fourier_magnitudes(after)
    if np.allclose(before, after):
        return True
    else:
        return False


In [ ]:
c_maj_profile = (1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1)
equivalence_comparison(c_maj_profile, operation="rotate")

In [ ]:
equivalence_comparison(c_maj_profile, operation="mirror")